# LinQ TPU: детекции YOLOv8s по тестовым роликам → JSONL для backend (DETECTOR=replay)

Запускается в Jupyter организаторов (http://hackathon.lin-q.ru/, нужен VPN). Результат — файлы
`dets_linq_<ролик>.jsonl` в формате FrameDetections (`detector_kind = "linq_tpu"`), которые:
- backend проигрывает с `DETECTOR=replay` и показывает как «LinQ TPU (запись)»;
- сравниваются с ONNX CPU: `linq/compare_with_onnx.py`; скорость — `linq/benchmark_linq.py`.

Пре- и постпроцессинг скопированы из `backend/app/pipeline/detectors/yolo_common.py` (те же letterbox и NMS).

**Ячейку «Вызов TPU» нужно заполнить по `examples/example.ipynb` в Jupyter** — API LinQ здесь не угадывается.
Пошагово — `docs/LINQ_TODO_FOR_HUMAN.md`.

In [ ]:
import json
import os
import time
from collections.abc import Iterable
from pathlib import Path

import cv2
import numpy as np

# !!! свой логин Jupyter (как в examples/example.ipynb); в репозиторий НЕ коммитить
LOGIN = os.environ.get("LINQ_LOGIN") or "hackathonXXXXX"
# Скомпилированная программа TPU (linq/build/), загруженная в Jupyter рядом с notebook
PROGRAM = "yolov8s_b1.tpu"
# Так в примере организаторов: программы должны лежать в /auto/cache/<LOGIN>
PROGRAM_DIR = os.path.join("/auto/cache", LOGIN)
PROGRAM_PATH = os.path.join(PROGRAM_DIR, PROGRAM)

# Загрузите ролики в Jupyter (кнопка Upload); имена — как в backend/data/videos/manifest.json
VIDEOS = {
    "te1": "0_te1.mp4",
    "te21": "0_te21.mp4",
    "kth_running": "person15_running_d1_uncomp.avi",
}
ANALYSIS_FPS = 10  # как analysis_fps источника в backend
INPUT_SIZE = 640
CONF_THRESHOLD = 0.1  # как DETECTOR_CONF_THRESHOLD backend (слабые детекции нужны ByteTrack)
IOU_THRESHOLD = 0.45
CLASS_IDS = {0}  # person
NAMES = {0: "person", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
MODEL_VERSION = f"yolov8s-linq:{PROGRAM}"

In [ ]:
# Копируем программу туда, где её ждёт сервис (так в examples/example.ipynb)
!mkdir -p {PROGRAM_DIR}
!cp {PROGRAM} {PROGRAM_DIR}
print(PROGRAM_PATH, os.path.exists(PROGRAM_PATH))

## Пре- и постпроцессинг (копия yolo_common.py)

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class LetterboxInfo:
    scale: float
    pad_x: int
    pad_y: int
    frame_w: int
    frame_h: int


def letterbox(
    image_bgr: np.ndarray, size: int = 640, pad_value: int = 114
) -> tuple[np.ndarray, LetterboxInfo]:
    """BGR uint8 (H, W, 3) -> тензор [1, 3, size, size] float32, RGB, 0..1.

    Масштаб с сохранением пропорций, кадр по центру, поля залиты pad_value (как в ultralytics).
    """
    h, w = image_bgr.shape[:2]
    scale = min(size / w, size / h)
    new_w, new_h = round(w * scale), round(h * scale)
    resized = cv2.resize(image_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    pad_x, pad_y = (size - new_w) // 2, (size - new_h) // 2
    canvas = np.full((size, size, 3), pad_value, dtype=np.uint8)
    canvas[pad_y : pad_y + new_h, pad_x : pad_x + new_w] = resized
    rgb = canvas[:, :, ::-1]
    tensor = np.ascontiguousarray(rgb.transpose(2, 0, 1)[None], dtype=np.float32) / 255.0
    return tensor, LetterboxInfo(scale, pad_x, pad_y, w, h)


def nms(boxes_xyxy: np.ndarray, scores: np.ndarray, iou_threshold: float) -> np.ndarray:
    """Классический NMS. Возвращает индексы оставшихся боксов по убыванию score."""
    order = scores.argsort()[::-1]
    x1, y1, x2, y2 = boxes_xyxy.T
    areas = (x2 - x1).clip(0) * (y2 - y1).clip(0)
    keep = []
    while order.size:
        i = order[0]
        keep.append(i)
        rest = order[1:]
        iw = (np.minimum(x2[i], x2[rest]) - np.maximum(x1[i], x1[rest])).clip(0)
        ih = (np.minimum(y2[i], y2[rest]) - np.maximum(y1[i], y1[rest])).clip(0)
        inter = iw * ih
        iou = inter / (areas[i] + areas[rest] - inter + 1e-9)
        order = rest[iou <= iou_threshold]
    return np.array(keep, dtype=np.int64)


def dequantize(q: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    """int8/uint8 -> float32: x = (q − zero_point) · scale. Параметры даёт компилятор LinQ."""
    return (q.astype(np.float32) - np.float32(zero_point)) * np.float32(scale)


def postprocess(
    output: np.ndarray,
    info: LetterboxInfo,
    names: dict[int, str],
    *,
    conf_threshold: float = 0.35,
    iou_threshold: float = 0.45,
    class_ids: Iterable[int] | None = None,
    max_det: int = 300,
) -> list[dict]:
    """Сырой выход YOLOv8 -> детекции в долях 0..1 исходного кадра.

    Шаги: транспонирование, лучший класс, порог, фильтр классов, cxcywh -> xyxy,
    NMS отдельно по каждому классу, снятие паддинга и масштаба, обрезка по кадру, нормализация.
    """
    pred = np.asarray(output, dtype=np.float32)
    if pred.ndim == 3:
        pred = pred[0]
    if pred.ndim != 2 or pred.shape[0] < 5:
        raise ValueError(f"Неожиданная форма выхода YOLO: {np.shape(output)}")
    pred = pred.T  # (N, 4 + C)

    class_scores = pred[:, 4:]
    cls = class_scores.argmax(axis=1)
    conf = class_scores[np.arange(len(cls)), cls]
    mask = conf >= conf_threshold
    if class_ids is not None:
        mask &= np.isin(cls, list(class_ids))
    if not mask.any():
        return []
    boxes, conf, cls = pred[mask, :4], conf[mask], cls[mask]

    cx, cy, bw, bh = boxes.T
    xyxy = np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], axis=1)

    # NMS по классам: сдвигаем боксы разных классов так, чтобы они не пересекались.
    offset = cls[:, None].astype(np.float32) * (xyxy.max() + 1.0)
    keep = nms(xyxy + offset, conf, iou_threshold)[:max_det]
    xyxy, conf, cls = xyxy[keep], conf[keep], cls[keep]

    xyxy[:, [0, 2]] = ((xyxy[:, [0, 2]] - info.pad_x) / info.scale).clip(0, info.frame_w)
    xyxy[:, [1, 3]] = ((xyxy[:, [1, 3]] - info.pad_y) / info.scale).clip(0, info.frame_h)
    xyxy[:, [0, 2]] /= info.frame_w
    xyxy[:, [1, 3]] /= info.frame_h

    return [
        dict(
            label=names.get(int(c), str(int(c))),
            confidence=round(float(s), 4),
            bbox=[round(float(v), 5) for v in box],
        )
        for box, s, c in zip(xyxy, conf, cls, strict=True)
        if box[2] - box[0] > 1e-4 and box[3] - box[1] > 1e-4
    ]

## Вызов TPU — ЗАПОЛНИТЬ по examples/example.ipynb

Нужно: загрузить программу `PROGRAM_PATH`, передать тензор `[1, 3, 640, 640]` и вернуть выход сети.
- Если вход программы int8 — квантовать параметрами компилятора: `q = round(x / scale) + zero_point`.
- Если выход int8 — `OUTPUT_QUANT = (scale, zero_point)` из отчёта компилятора, иначе `None`.
- Выход должен стать float32 `[1, 84, 8400]` (или сырые головы, если граф отрезан — тогда декодирование на CPU).
- Если API отдаёт время на самом TPU — верните его вторым значением (иначе `None`).

In [ ]:
OUTPUT_QUANT = None  # (scale, zero_point) выхода, если он int8


def run_tpu(tensor: np.ndarray):
    """-> (сырой выход сети, время на TPU в мс или None). ЗАПОЛНИТЬ по examples/example.ipynb."""
    raise NotImplementedError("Перенесите сюда вызов программы TPU из examples/example.ipynb")


def to_float(raw) -> np.ndarray:
    arr = np.asarray(raw)
    if np.issubdtype(arr.dtype, np.integer):
        assert OUTPUT_QUANT is not None, "выход int8: задайте OUTPUT_QUANT = (scale, zero_point)"
        return dequantize(arr, *OUTPUT_QUANT)
    return arr.astype(np.float32)

## Прогон роликов → JSONL

In [ ]:
def run_video(video_id: str, path: str) -> Path:
    cap = cv2.VideoCapture(path)
    native = cap.get(cv2.CAP_PROP_FPS) or 25
    step, next_ts, idx = 1 / ANALYSIS_FPS, 0.0, -1
    out = Path(f"dets_linq_{video_id}.jsonl")
    with out.open("w", encoding="utf-8") as fh:
        while cap.grab():
            idx += 1
            ts = idx / native
            if ts + 1e-3 < next_ts:
                continue
            ok, img = cap.retrieve()
            if not ok:
                break
            next_ts += step
            t0 = time.perf_counter()
            tensor, info = letterbox(img, INPUT_SIZE)
            t1 = time.perf_counter()
            raw, device_ms = run_tpu(tensor)
            t2 = time.perf_counter()
            dets = postprocess(
                to_float(raw),
                info,
                NAMES,
                conf_threshold=CONF_THRESHOLD,
                iou_threshold=IOU_THRESHOLD,
                class_ids=CLASS_IDS,
            )
            t3 = time.perf_counter()
            fh.write(
                json.dumps(
                    {
                        "frame_idx": idx,
                        "video_ts": round(ts, 4),
                        "frame_w": img.shape[1],
                        "frame_h": img.shape[0],
                        "inference_ms": round((t2 - t1) * 1000, 3),
                        "preprocess_ms": round((t1 - t0) * 1000, 3),
                        "postprocess_ms": round((t3 - t2) * 1000, 3),
                        "device_ms": device_ms,
                        "model_version": MODEL_VERSION,
                        "detector_kind": "linq_tpu",
                        "detections": dets,
                    }
                )
                + "\n"
            )
    cap.release()
    return out


results = {vid: run_video(vid, path) for vid, path in VIDEOS.items() if os.path.exists(path)}
results

## Сводка замеров (то же считает linq/benchmark_linq.py)

In [ ]:
for vid, path in results.items():
    with open(path, encoding="utf-8") as fh:
        recs = [json.loads(line) for line in fh][5:]  # без прогрева
    inf = np.array([r["inference_ms"] for r in recs])
    pre = np.array([r["preprocess_ms"] for r in recs])
    post = np.array([r["postprocess_ms"] for r in recs])
    print(
        f"{vid}: кадров {len(recs)}, инференс avg {inf.mean():.1f} "
        f"p50 {np.percentile(inf, 50):.1f} "
        f"p95 {np.percentile(inf, 95):.1f} мс; пре {pre.mean():.1f}, пост {post.mean():.1f} мс; "
        f"FPS ≈ {1000 / (inf.mean() + pre.mean() + post.mean()):.1f}"
    )